# iRage AlgoArena — Short-Horizon Return Prediction
## Final Submission Notebook (v11 Kaggle)

**v11 = v10 + CatBoost restored + LGB_Huber lr reverted + LGB gain selection + more features**

| Component | v10 Kaggle | v11 Kaggle |
|---|---|---|
| Feature selection | ~13s (Pearson corr) | **~60s** (LGB gain, 1 fold × 80 rounds) |
| N features | 150 | **200** (more signal) |
| LGB_Huber lr | 0.05 (~420 iter) | **0.02** (~1048 iter, v8 value) |
| LGB_Shallow | 0.05, 5 folds | 0.05, 5 folds (unchanged) |
| CatBoost | **dropped** | **restored** (depth=5, od_wait=50) |
| Runtime | 10.3 min | **~27 min** (target 25-28 min) |

**Why these changes:**
- v10 spent only 10.3 of 30 allowed minutes — 19.7 min wasted budget
- LGB gain selection captures feature interactions; Pearson only captures linear signal
- lr=0.02 on LGB_Huber was v8 best (0.001956 OOF); lr=0.05 converges faster but shallower
- CatBoost adds ensemble diversity + is regime-stable (never went negative in v8)
- 200 features (vs 150) captures more signal without blowing the budget

**Time guards (critical):**
- `HUBER_TIME_LIMIT = 850s` — LGB_Huber stops here; Shallow + CAT get the rest
- `SHAL_TIME_LIMIT  = 1100s` — LGB_Shallow stops here; CAT gets ~580s
- `CAT_HARD_STOP    = 1650s` — CAT stops 30s before notebook kill
- `HARD_STOP_S      = 1680s` — absolute notebook ceiling (28 min)

**Compliance:** unchanged from v10 — all row-wise FE, train-only normalization, GroupKFold.

In [ ]:
# CELL 1 - IMPORTS
import os, gc, warnings, time
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
from pathlib import Path
from scipy.optimize import minimize

import lightgbm as lgb
from sklearn.model_selection import GroupKFold, KFold
from sklearn.metrics import r2_score

try:
    from catboost import CatBoostRegressor, Pool
    CATBOOST_AVAILABLE = True
    print('CatBoost OK')
except ImportError:
    CATBOOST_AVAILABLE = False
    print('CatBoost not available — will run LGB only')

SEED = 23
np.random.seed(SEED)
NOTEBOOK_START = time.time()

print(f'LightGBM {lgb.__version__}')
print('Imports OK')

In [ ]:
# CELL 2 - CONFIG
class CFG:
    DATA_DIR   = Path('/kaggle/input/competitions/short-horizon-return-prediction-challenge-by-i-rage')
    TRAIN_PATH = DATA_DIR / 'train.parquet'
    TEST_PATH  = DATA_DIR / 'test.parquet'
    OUTPUT_DIR = Path('/kaggle/working')
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

    N_FOLDS = 5
    SEED    = 23

    CLIP_LOWER = 2
    CLIP_UPPER = 98

    BUILD_MOMENTUM     = True
    BUILD_ABS_MOMENTUM = True
    BUILD_PRICE_FEATS  = True
    BUILD_FAMILY_AGG   = True
    BUILD_SO3_POLY     = True

    # v11: 200 features (was 150). Captures more signal within budget.
    # Selected by LGB gain (1 fold x 80 rounds) instead of Pearson corr.
    N_TOP_FEATURES = 200

    STANDARDIZE = True

    # ── LGB_Huber ─────────────────────────────────────────────────
    # v11: lr=0.02 (reverted to v8 value). v8 had avg best_iter=1048
    # at this lr. v10's lr=0.05 converged faster (~420 iter) but
    # v8 produced better OOF R2 (0.001956 vs v10's unknown).
    # With 200 features (vs 300 in v8) this is ~33% faster per fold.
    LGB_HUBER_PARAMS = {
        'objective':         'huber',
        'alpha':             0.02,
        'metric':            'huber',
        'learning_rate':     0.02,   # v11: reverted from 0.05 to v8's value
        'num_leaves':        15,
        'max_depth':         -1,
        'min_child_samples': 200,
        'feature_fraction':  0.6,
        'bagging_fraction':  0.8,
        'bagging_freq':      1,
        'lambda_l1':         0.5,
        'lambda_l2':         8.0,
        'min_gain_to_split': 0.0,
        'verbose':           -1,
        'n_jobs':            -1,
        'seed':              SEED,
    }

    # ── LGB_Shallow ───────────────────────────────────────────────
    # Unchanged from v10. 7 leaves = global patterns only.
    LGB_SHALLOW_PARAMS = {
        'objective':         'huber',
        'alpha':             0.02,
        'metric':            'huber',
        'learning_rate':     0.05,
        'num_leaves':        7,
        'max_depth':         -1,
        'min_child_samples': 500,
        'feature_fraction':  0.5,
        'bagging_fraction':  0.8,
        'bagging_freq':      1,
        'lambda_l1':         1.0,
        'lambda_l2':         15.0,
        'min_gain_to_split': 0.0,
        'verbose':           -1,
        'n_jobs':            -1,
        'seed':              SEED + 2,
    }

    LGB_NUM_ROUNDS  = 3000
    LGB_EARLY_STOP  = 100
    SHAL_EARLY_STOP = 80

    # ── CatBoost (restored in v11) ─────────────────────────────────
    # v8 params: OOF R2=0.001451 (best single model), avg_iter=182.
    # Never went negative across any fold (ordered boosting protects).
    # od_wait=50: stops 50 rounds after val improvement plateau.
    # iterations=600: hard cap (od_wait fires at ~232 avg, so 600 is safe).
    CAT_PARAMS = dict(
        loss_function='Huber:delta=0.05',
        eval_metric='RMSE',
        iterations=600,
        learning_rate=0.04,
        depth=5,
        l2_leaf_reg=10.0,
        min_data_in_leaf=150,
        border_count=128,
        random_seed=SEED,
        od_type='Iter',
        od_wait=50,
        verbose=200,
        task_type='CPU',
        thread_count=-1,
        allow_writing_files=False,
    )

    # ── Time guards ───────────────────────────────────────────────
    # v11 budget math (Kaggle timings):
    #   Setup (load+FE+zscore+LGB_sel): ~160s
    #   LGB_Huber 5f (lr=0.02, 200 feat): ~625s  -> stop at 850s elapsed
    #   LGB_Shallow 5f (lr=0.05, 200 feat): ~200s -> stop at 1100s elapsed
    #   CatBoost 5f (od_wait=50): ~500s           -> stop at 1650s elapsed
    #   Ensemble + save: ~50s
    #   Total estimate: ~1585s = 26.4 min (target: 25-28 min)
    HUBER_TIME_LIMIT = 850    # LGB_Huber stops here
    SHAL_TIME_LIMIT  = 1100   # LGB_Shallow stops here
    CAT_HARD_STOP    = 1650   # CatBoost stops here (30s before kill)
    HARD_STOP_S      = 1680   # absolute hard stop (28 min)

print('Config OK')
print(f'N_FOLDS={CFG.N_FOLDS} | N_TOP_FEATURES={CFG.N_TOP_FEATURES}')
print(f'LGB_Huber lr={CFG.LGB_HUBER_PARAMS["learning_rate"]} | LGB_Shallow lr={CFG.LGB_SHALLOW_PARAMS["learning_rate"]}')
print(f'Time guards: Huber={CFG.HUBER_TIME_LIMIT}s | Shallow={CFG.SHAL_TIME_LIMIT}s | CAT={CFG.CAT_HARD_STOP}s')

In [ ]:
# CELL 3 - DATA LOADING
print('Loading data...')
t0 = time.time()

train = pd.read_parquet(CFG.TRAIN_PATH)
test  = pd.read_parquet(CFG.TEST_PATH)

print(f'Train: {train.shape} | Test: {test.shape}')
print(f'Loaded in {time.time()-t0:.1f}s')

train_ids = train['ID'].values.copy()
test_ids  = test['ID'].values.copy()
y_raw     = train['TARGET'].copy()

if 'CV_GROUP' in train.columns:
    cv_groups    = train['CV_GROUP'].values.copy()
    USE_GROUP_CV = True
    n_groups     = len(np.unique(cv_groups))
    print(f'CV_GROUP: {n_groups} groups -> GroupKFold({CFG.N_FOLDS})')
    train.drop(columns=['ID', 'TARGET', 'CV_GROUP'], inplace=True)
else:
    cv_groups    = None
    USE_GROUP_CV = False
    print('No CV_GROUP -> KFold fallback')
    train.drop(columns=['ID', 'TARGET'], inplace=True)

test.drop(columns=['ID'], inplace=True)

assert list(train.columns) == list(test.columns), 'FATAL: column mismatch!'
print(f'Features: {train.shape[1]} | Elapsed: {time.time()-NOTEBOOK_START:.0f}s')

In [ ]:
# CELL 4 - TARGET CLIPPING
# P2/P98 clip confirmed better than P1/P99 in v5/v6 comparison.
# kurtosis=48 justifies Huber loss: extreme returns are noise, not signal.
# TRAINING ONLY — test predictions are unconstrained.
desc = y_raw.describe(percentiles=[0.01, 0.02, 0.05, 0.25, 0.5, 0.75, 0.95, 0.98, 0.99])
print(desc)
print(f'Kurtosis: {y_raw.kurtosis():.2f} (fat tails -> Huber loss justified)')

clip_lo  = y_raw.quantile(CFG.CLIP_LOWER / 100)
clip_hi  = y_raw.quantile(CFG.CLIP_UPPER / 100)
y_series = y_raw.clip(clip_lo, clip_hi)
n_clip   = int(((y_raw < clip_lo) | (y_raw > clip_hi)).sum())
print(f'Clip P{CFG.CLIP_LOWER}/P{CFG.CLIP_UPPER}: [{clip_lo:.6f}, {clip_hi:.6f}]')
print(f'Clipped: {n_clip:,} rows ({100*n_clip/len(y_raw):.2f}%) | Std: {y_raw.std():.4f} -> {y_series.std():.4f}')

y = y_series.values.astype(np.float32)
print('Target ready OK')

In [ ]:
# CELL 5 - FEATURE CATEGORIZATION
all_cols  = train.columns.tolist()
lag1_cols = [c for c in all_cols if c.endswith('_LagT1')]
lag2_cols = [c for c in all_cols if c.endswith('_LagT2')]
lag3_cols = [c for c in all_cols if c.endswith('_LagT3')]
lag_set   = set(lag1_cols + lag2_cols + lag3_cols)
base_cols = [c for c in all_cols if c not in lag_set]

special_cols   = ['Price', 'SO3_T']
pure_base_cols = [c for c in base_cols if c not in special_cols]

def get_family(col):
    return col.split('_')[0]

families = sorted(set(get_family(c) for c in pure_base_cols))

lag_map = {
    c: {
        'lag1': c+'_LagT1' if c+'_LagT1' in lag_set else None,
        'lag2': c+'_LagT2' if c+'_LagT2' in lag_set else None,
        'lag3': c+'_LagT3' if c+'_LagT3' in lag_set else None,
    } for c in pure_base_cols
}

print(f'Pure base: {len(pure_base_cols)} | Lag1/2/3: {len(lag1_cols)}/{len(lag2_cols)}/{len(lag3_cols)}')
print(f'Families: {families} | Total: {len(all_cols)}')

In [ ]:
# CELL 6 - FEATURE ENGINEERING
# ALL operations: purely row-wise (element-wise or row-axis aggregations).
# No cross-sample operations on train or test.
# Identical function applied to both — no information leakage.

def feature_engineering(df, lag_map, families, pure_base_cols, verbose=True):
    df = df.copy()
    new_feats = []

    # A. MOMENTUM: acceleration between lag windows
    # Acc12 = LagT1 - LagT2: recent acceleration
    # Acc23 = LagT2 - LagT3: medium-term acceleration
    # Div13 = LagT1 - LagT3: total divergence
    if CFG.BUILD_MOMENTUM:
        n0 = len(new_feats)
        for bc, lags in lag_map.items():
            l1, l2, l3 = lags['lag1'], lags['lag2'], lags['lag3']
            if l1 and l2 and l1 in df.columns and l2 in df.columns:
                df[bc+'_Acc12'] = df[l1] - df[l2]; new_feats.append(bc+'_Acc12')
            if l2 and l3 and l2 in df.columns and l3 in df.columns:
                df[bc+'_Acc23'] = df[l2] - df[l3]; new_feats.append(bc+'_Acc23')
            if l1 and l3 and l1 in df.columns and l3 in df.columns:
                df[bc+'_Div13'] = df[l1] - df[l3]; new_feats.append(bc+'_Div13')
        if verbose: print(f'  Momentum: {len(new_feats)-n0}')

    # B. |LagT1|: signed momentum magnitude
    # Trees cannot represent |x| from x alone (requires two splits).
    # Explicitly providing |LagT1| doubles the information per lag feature.
    if CFG.BUILD_ABS_MOMENTUM:
        n0 = len(new_feats)
        for bc, lags in lag_map.items():
            l1 = lags['lag1']
            if l1 and l1 in df.columns:
                df[bc+'_AbsMom'] = df[l1].abs(); new_feats.append(bc+'_AbsMom')
        if verbose: print(f'  AbsMom:   {len(new_feats)-n0}')

    # C. Price-specific features
    if CFG.BUILD_PRICE_FEATS and 'Price' in df.columns:
        n0 = len(new_feats)
        pl1, pl2, pl3 = 'Price_LagT1', 'Price_LagT2', 'Price_LagT3'
        if pl1 in df.columns and pl2 in df.columns:
            df['Price_Acc']    = df[pl1] - df[pl2]; new_feats.append('Price_Acc')
        if pl1 in df.columns and pl3 in df.columns:
            df['Price_Div13']  = df[pl1] - df[pl3]; new_feats.append('Price_Div13')
        if pl2 in df.columns and pl3 in df.columns:
            df['Price_Acc23']  = df[pl2] - df[pl3]; new_feats.append('Price_Acc23')
        if pl1 in df.columns:
            df['Price_AbsMom'] = df[pl1].abs();      new_feats.append('Price_AbsMom')
        if verbose: print(f'  Price:    {len(new_feats)-n0}')

    # D. Family aggregates: cross-instrument consensus/dispersion
    # BaseMean/Std: cross-section level + dispersion
    # MomMean/Std/Spread: cross-section momentum consensus
    if CFG.BUILD_FAMILY_AGG:
        n0 = len(new_feats)
        for fam in families:
            fb  = [c for c in pure_base_cols if get_family(c)==fam and c in df.columns]
            fl1 = [c+'_LagT1' for c in fb if c+'_LagT1' in df.columns]
            if len(fb) < 2: continue
            df[fam+'_BaseMean']  = df[fb].mean(axis=1);  new_feats.append(fam+'_BaseMean')
            df[fam+'_BaseStd']   = df[fb].std(axis=1);   new_feats.append(fam+'_BaseStd')
            if len(fl1) >= 2:
                fmom = df[fl1]
                df[fam+'_MomMean']   = fmom.mean(axis=1); new_feats.append(fam+'_MomMean')
                df[fam+'_MomStd']    = fmom.std(axis=1);  new_feats.append(fam+'_MomStd')
                df[fam+'_MomSpread'] = fmom.max(axis=1) - fmom.min(axis=1)
                new_feats.append(fam+'_MomSpread')
        if verbose: print(f'  FamilyAgg:{len(new_feats)-n0}')

    # E. SO3_T polynomial + family interactions
    # SO3_T is the only named covariate. Polynomial terms and interactions
    # with family momentum capture nonlinear regime effects.
    if CFG.BUILD_SO3_POLY and 'SO3_T' in df.columns:
        n0 = len(new_feats)
        df['SO3_T_sq'] = df['SO3_T']**2; new_feats.append('SO3_T_sq')
        df['SO3_T_cb'] = df['SO3_T']**3; new_feats.append('SO3_T_cb')
        if 'Price_LagT1' in df.columns:
            df['SO3T_x_PriceMom'] = df['SO3_T'] * df['Price_LagT1']
            new_feats.append('SO3T_x_PriceMom')
        for fam in families:
            fn = fam+'_MomMean'
            if fn in df.columns:
                df['SO3T_x_'+fam+'_Mom'] = df['SO3_T'] * df[fn]
                new_feats.append('SO3T_x_'+fam+'_Mom')
        if verbose: print(f'  SO3_T:    {len(new_feats)-n0}')

    if verbose: print(f'  Total new: {len(new_feats)} | Grand total: {df.shape[1]}')
    return df, new_feats


print('Running FE...')
t0 = time.time()
train_fe, new_feats = feature_engineering(train, lag_map, families, pure_base_cols)
test_fe,  _         = feature_engineering(test,  lag_map, families, pure_base_cols, verbose=False)
print(f'Done in {time.time()-t0:.1f}s | Train: {train_fe.shape} | Test: {test_fe.shape}')

In [ ]:
# CELL 7 - FINALIZE ARRAYS
train_fe.replace([np.inf, -np.inf], np.nan, inplace=True)
test_fe.replace( [np.inf, -np.inf], np.nan, inplace=True)
assert list(train_fe.columns) == list(test_fe.columns), 'FATAL: column mismatch!'

feature_cols = train_fe.columns.tolist()
n_raw_features = len(feature_cols)
print(f'Total features before selection: {n_raw_features}')

X      = train_fe.values.astype(np.float32)
X_test = test_fe.values.astype(np.float32)
print(f'X: {X.shape} | X_test: {X_test.shape}')

del train_fe, test_fe, train, test
gc.collect()
print(f'Memory freed | Elapsed: {time.time()-NOTEBOOK_START:.0f}s')

In [ ]:
# CELL 8 - Z-SCORE STANDARDIZATION (TRAIN STATS ONLY)
# Must happen before feature selection so |corr| is computed
# on the same normalized scale the models will train on.
# COMPLIANCE: only train mean/std used. Test is transformed
# with train statistics (no test-set information).

if CFG.STANDARDIZE:
    t_std   = time.time()
    tr_mean = np.nanmean(X, axis=0).astype(np.float32)
    tr_std  = np.nanstd(X,  axis=0).astype(np.float32)
    tr_std  = np.where(tr_std < 1e-8, 1.0, tr_std)

    X      = np.clip((X      - tr_mean) / tr_std, -10, 10).astype(np.float32)
    X_test = np.clip((X_test - tr_mean) / tr_std, -10, 10).astype(np.float32)
    X      = np.nan_to_num(X,      nan=0.0)
    X_test = np.nan_to_num(X_test, nan=0.0)
    print(f'Standardized | mean={X.mean():.4f} std={X.std():.4f} | {time.time()-t_std:.1f}s')

print(f'Arrays: X={X.shape} X_test={X_test.shape} | Elapsed: {time.time()-NOTEBOOK_START:.0f}s')

In [ ]:
# CELL 9 - FEATURE SELECTION (LGB GAIN — 1 FOLD × 80 ROUNDS)
#
# v11 replaces Pearson correlation with LGB gain importance.
# WHY: Pearson corr only captures LINEAR marginal signal.
#   A feature important via interactions (e.g. SO3_T × MomMean) has
#   near-zero Pearson corr but high gain in a tree model.
#   v8 used 500-round LGB (193s local). v11 uses 80-round (fast enough
#   to detect presence, not precise gain magnitude).
#
# COST: 1 fold × 80 rounds × (same params as main model = fast)
#   ≈ 60s on Kaggle (vs 13s for Pearson, vs 476s for v8's 500-round pass).
#
# POLICY: training data only. No test information. Row-wise features only.

print(f'Feature selection: {n_raw_features} -> top {CFG.N_TOP_FEATURES} by LGB gain...')
t_fs = time.time()

# Use fold 0 of GroupKFold for selection (same split as main CV)
if USE_GROUP_CV:
    _spl_fs = list(GroupKFold(n_splits=CFG.N_FOLDS).split(X, y, groups=cv_groups))
else:
    _spl_fs = list(KFold(n_splits=CFG.N_FOLDS, shuffle=True, random_state=CFG.SEED).split(X, y))
trn_fs, val_fs = _spl_fs[0]

# Same params as main model — no num_leaves override (was the 900s bug in v11 local)
_fs_params = {
    **CFG.LGB_HUBER_PARAMS,
    'learning_rate': 0.05,   # faster convergence for selection pass
    # num_leaves: inherits 15 from LGB_HUBER_PARAMS (critical — do NOT override to 31)
}

_d_trn = lgb.Dataset(X[trn_fs], label=y[trn_fs], feature_name=feature_cols, free_raw_data=True)
# No val set: goal is gain>0 presence, not val metric convergence
_m = lgb.train(
    _fs_params, _d_trn,
    num_boost_round=80,            # 80 rounds sufficient for presence detection
    callbacks=[lgb.log_evaluation(-1)],
)
imp_gain = _m.feature_importance(importance_type='gain').astype(np.float64)
del _m, _d_trn
gc.collect()

# Rank by gain descending, take top N
rank_idx       = np.argsort(imp_gain)[::-1]
sel_idx        = rank_idx[:CFG.N_TOP_FEATURES]
sel_idx_sorted = np.sort(sel_idx)
feature_cols   = [feature_cols[i] for i in sel_idx_sorted]
n_features     = len(feature_cols)

X      = X[:,      sel_idx_sorted]
X_test = X_test[:, sel_idx_sorted]

n_zero = (imp_gain == 0).sum()
print(f'Selection done in {time.time()-t_fs:.1f}s')
print(f'Selected: {n_features} features from {n_raw_features} | Zero-gain: {n_zero}')
print(f'Top-5 by gain: {feature_cols[:5]}')
print(f'Final: X={X.shape} X_test={X_test.shape} | Elapsed: {time.time()-NOTEBOOK_START:.0f}s')

del imp_gain, rank_idx, sel_idx, sel_idx_sorted, _spl_fs
gc.collect()

In [ ]:
# CELL 10 - CV SPLITTER
if USE_GROUP_CV:
    splitter = GroupKFold(n_splits=CFG.N_FOLDS)
    splits   = list(splitter.split(X, y, groups=cv_groups))
    print(f'GroupKFold({CFG.N_FOLDS}) on CV_GROUP ({n_groups} groups)')
    for i, (ti, vi) in enumerate(splits):
        g_in_val = len(np.unique(cv_groups[vi]))
        print(f'  Fold {i+1}: {len(ti):,} train | {len(vi):,} val | {g_in_val} groups')
else:
    splits = list(KFold(CFG.N_FOLDS, shuffle=True, random_state=CFG.SEED).split(X, y))
    print(f'KFold({CFG.N_FOLDS}) fallback')
print('CV ready OK')

In [ ]:
# CELL 11 - LGB TRAINER
# Time-aware: checks elapsed before each fold.
# If remaining budget for this model is < 60s, stops early.
# Incomplete folds: test predictions rescaled proportionally.

def run_lgb_cv(params, num_rounds, early_stop, splits, X, y, X_test,
               feature_cols, name='LGB', time_limit_s=CFG.HARD_STOP_S):
    oof    = np.zeros(len(y),       dtype=np.float64)
    ptest  = np.zeros(len(X_test),  dtype=np.float64)
    scores, iters = [], []
    n_done = 0

    print(f'--- {name} ({len(splits)} folds, limit={time_limit_s}s) ---')

    for fold, (ti, vi) in enumerate(splits):
        elapsed = time.time() - NOTEBOOK_START
        # Stop if we've hit the per-model time limit
        if elapsed > time_limit_s:
            print(f'  [{name}] Time limit {time_limit_s}s reached at fold {fold+1} (elapsed={elapsed:.0f}s). Stopping.')
            break

        t0   = time.time()
        dtr  = lgb.Dataset(X[ti], label=y[ti], feature_name=feature_cols, free_raw_data=True)
        dval = lgb.Dataset(X[vi], label=y[vi], reference=dtr,             free_raw_data=True)

        m = lgb.train(
            params, dtr,
            num_boost_round=num_rounds,
            valid_sets=[dtr, dval],
            valid_names=['train', 'valid'],
            callbacks=[
                lgb.early_stopping(early_stop, verbose=False),
                lgb.log_evaluation(500),
            ],
        )
        bi = m.best_iteration
        iters.append(bi)

        vp        = m.predict(X[vi],   num_iteration=bi)
        oof[vi]   = vp
        ptest    += m.predict(X_test,  num_iteration=bi) / len(splits)

        fr2 = r2_score(y[vi], vp)
        scores.append(fr2)
        n_done += 1
        fold_t = time.time() - t0
        print(f'  Fold {fold+1}/{len(splits)} | R2: {fr2:+.6f} | iter: {bi} | {fold_t:.0f}s | total: {time.time()-NOTEBOOK_START:.0f}s')

        del m, dtr, dval
        gc.collect()

    if n_done == 0:
        print(f'  WARNING: {name} — zero folds completed. Returning zeros.')
        return oof, ptest, iters, scores, 0.0

    if n_done < len(splits):
        # Rescale: we averaged over len(splits) but only ran n_done folds
        ptest  = ptest * len(splits) / n_done
        fi     = np.concatenate([splits[i][1] for i in range(n_done)])
        oof_r2 = r2_score(y[fi], oof[fi])
        print(f'  {name} PARTIAL ({n_done}/{len(splits)} folds) OOF R2: {oof_r2:+.6f}')
    else:
        oof_r2 = r2_score(y, oof)
        print(f'  {name} OOF R2: {oof_r2:+.6f} | fold std: {np.std(scores):.6f} | avg_iter: {np.mean(iters):.0f}')

    return oof, ptest, iters, scores, oof_r2

print('LGB trainer defined OK')

In [ ]:
# CELL 12 - RUN LGB MODELS
print('='*60)
print(f'LightGBM CV — {CFG.N_FOLDS} folds × 2 models')
print(f'v11: lr=0.02/0.05, 200 features, LGB gain selection, CAT restored')
print('='*60)

# LGB_Huber: stop at 850s so Shallow + CAT have room
oof_lgb_huber, pred_lgb_huber, iters_huber, scores_huber, r2_lgb_huber = run_lgb_cv(
    CFG.LGB_HUBER_PARAMS, CFG.LGB_NUM_ROUNDS, CFG.LGB_EARLY_STOP,
    splits, X, y, X_test, feature_cols,
    name='LGB_Huber',
    time_limit_s=CFG.HUBER_TIME_LIMIT,
)
print(f'LGB_Huber done | Elapsed: {time.time()-NOTEBOOK_START:.0f}s')

# LGB_Shallow: stop at 1100s so CAT gets ~550s
oof_lgb_shal, pred_lgb_shal, iters_shal, scores_shal, r2_lgb_shal = run_lgb_cv(
    CFG.LGB_SHALLOW_PARAMS, CFG.LGB_NUM_ROUNDS, CFG.SHAL_EARLY_STOP,
    splits, X, y, X_test, feature_cols,
    name='LGB_Shallow',
    time_limit_s=CFG.SHAL_TIME_LIMIT,
)
print(f'Both LGB models done | Elapsed: {time.time()-NOTEBOOK_START:.0f}s')

In [ ]:
# CELL 12b - CATBOOST WITH HUBER LOSS (restored in v11)
# v8 OOF R2: 0.001451 (best single model). Ordered boosting = never negative.
# od_wait=50: stops when val Huber hasn't improved for 50 rounds.
# Time guard: stops if NOTEBOOK_START + CAT_HARD_STOP is exceeded.
# FIX: no iter_budget formula (that caused bugs in v10-v12 local). Direct params.

oof_cat    = np.zeros(len(y),      dtype=np.float64)
pred_cat   = np.zeros(len(X_test), dtype=np.float64)
cat_scores, cat_iters = [], []
CATBOOST_DONE = False

if CATBOOST_AVAILABLE:
    print('='*55)
    print(f'CatBoost Huber — {CFG.N_FOLDS} folds')
    print('='*55)

    for fold, (ti, vi) in enumerate(splits):
        elapsed   = time.time() - NOTEBOOK_START
        remaining = CFG.CAT_HARD_STOP - elapsed
        print(f'  Fold {fold+1} | {remaining:.0f}s remaining (CAT budget)')

        if remaining < 90:
            print('  CAT time exhausted — stopping')
            break

        t0      = time.time()
        pool_tr = Pool(X[ti], y[ti])
        pool_va = Pool(X[vi], y[vi])

        m = CatBoostRegressor(**CFG.CAT_PARAMS)
        m.fit(pool_tr, eval_set=pool_va, use_best_model=True)

        vp          = m.predict(X[vi])
        oof_cat[vi] = vp
        pred_cat   += m.predict(X_test) / CFG.N_FOLDS

        fr2 = r2_score(y[vi], vp)
        bi  = m.get_best_iteration()
        if bi is None: bi = m.tree_count_ - 1   # fallback: prevent nan
        cat_scores.append(fr2)
        cat_iters.append(bi)
        print(f'  Fold {fold+1}/{CFG.N_FOLDS} | R2: {fr2:+.6f} | best_iter: {bi} | {time.time()-t0:.0f}s')

        del m, pool_tr, pool_va
        gc.collect()

    n_done = len(cat_scores)
    if n_done == 0:
        cat_r2 = 0.0
        pred_cat = np.zeros(len(X_test))
        CATBOOST_DONE = False
    elif n_done < CFG.N_FOLDS:
        pred_cat  = pred_cat * CFG.N_FOLDS / n_done
        fi        = np.concatenate([splits[i][1] for i in range(n_done)])
        cat_r2    = r2_score(y[fi], oof_cat[fi])
        CATBOOST_DONE = (n_done >= 3)   # only use if 3+ folds
        print(f'CAT partial ({n_done}/{CFG.N_FOLDS}) OOF R2: {cat_r2:+.6f}')
    else:
        cat_r2 = r2_score(y, oof_cat)
        CATBOOST_DONE = True
        print(f'CAT OOF R2: {cat_r2:+.6f} | std: {np.std(cat_scores):.6f} | avg_iter: {np.mean(cat_iters):.0f}')

    print(f'Elapsed: {time.time()-NOTEBOOK_START:.0f}s')
else:
    cat_r2 = 0.0
    print('CatBoost not available')


In [ ]:
# CELL 13 - ENSEMBLE OPTIMIZATION
# OOF-based Nelder-Mead weight optimization.
# Softmax weights: always sum to 1, always positive.
# No test-set information used in optimization.

model_names = ['LGB_Huber', 'LGB_Shallow']
oofs_list   = [oof_lgb_huber, oof_lgb_shal]
preds_list  = [pred_lgb_huber, pred_lgb_shal]

if CATBOOST_DONE:
    model_names.append('CAT_Huber')
    oofs_list.append(oof_cat)
    preds_list.append(pred_cat)
    print(f'Ensemble: 3 models (LGB_Huber + LGB_Shallow + CAT)')
else:
    print(f'Ensemble: 2 models (LGB_Huber + LGB_Shallow) — CAT excluded')

print('ENSEMBLE OPTIMIZATION')
print('-'*50)
for nm, oof in zip(model_names, oofs_list):
    print(f'  {nm:<14} OOF R2: {r2_score(y, oof):+.6f}')

# Simple average baseline
oof_sa  = np.mean(oofs_list, axis=0)
pred_sa = np.mean(preds_list, axis=0)
r2_sa   = r2_score(y, oof_sa)
print(f'Simple avg:  {r2_sa:+.6f}')

# Optimized weights via Nelder-Mead
oofs_arr  = np.stack(oofs_list, axis=1)   # (N, M)
preds_arr = np.stack(preds_list, axis=1)  # (N_test, M)

def neg_r2(rw):
    w = np.exp(rw) / np.exp(rw).sum()
    return -r2_score(y, (oofs_arr * w).sum(axis=1))

best_r2_opt, best_w = -np.inf, None
for s in range(15):
    rng = np.random.RandomState(s)
    w0  = rng.randn(len(oofs_list)) * 0.1
    res = minimize(neg_r2, w0, method='Nelder-Mead',
                   options={'maxiter': 3000, 'xatol': 1e-7, 'fatol': 1e-9})
    if -res.fun > best_r2_opt:
        best_r2_opt = -res.fun
        best_w      = np.exp(res.x) / np.exp(res.x).sum()

oof_opt  = (oofs_arr  * best_w).sum(axis=1)
pred_opt = (preds_arr * best_w).sum(axis=1)
r2_opt   = r2_score(y, oof_opt)

print(f'Optimized:   {r2_opt:+.6f}')
for nm, w in zip(model_names, best_w):
    print(f'  Weight {nm}: {w:.4f}')

if r2_opt > r2_sa:
    final_oof, final_pred = oof_opt, pred_opt
    final_r2, method_str  = r2_opt, 'Optimized wts'
else:
    final_oof, final_pred = oof_sa, pred_sa
    final_r2, method_str  = r2_sa, 'Simple avg'

print(f'FINAL: {final_r2:+.6f} ({method_str})')

In [ ]:
# CELL 14 - SUBMISSION GUARDS + SAVE
CFG.OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

submission = pd.DataFrame({'ID': test_ids, 'TARGET': final_pred})

assert len(submission)            == len(test_ids),   'FATAL: row count mismatch'
assert list(submission.columns)  == ['ID', 'TARGET'], 'FATAL: column names wrong'
assert submission['ID'].nunique() == len(test_ids),   'FATAL: duplicate IDs'
assert not submission['TARGET'].isnull().any(),        'FATAL: NaN in TARGET'
assert not np.isinf(submission['TARGET'].values).any(),'FATAL: Inf in TARGET'
assert submission['TARGET'].std() > 0,                'FATAL: constant TARGET'

print('All 6 guards passed OK')

sub_path = CFG.OUTPUT_DIR / 'submission.csv'
submission.to_csv(sub_path, index=False)

v = pd.read_csv(sub_path)
assert len(v) == len(test_ids), 'FATAL: CSV row count wrong'
assert list(v.columns) == ['ID', 'TARGET'], 'FATAL: CSV columns wrong'

total_elapsed = time.time() - NOTEBOOK_START
print(f'submission.csv SAVED | {len(submission):,} rows | {total_elapsed:.0f}s = {total_elapsed/60:.1f} min')
print(submission['TARGET'].describe())

In [ ]:
# CELL 15 - FINAL SCORECARD
total_elapsed = time.time() - NOTEBOOK_START

print('='*70)
print('  R2 SCORECARD — iRage Short-Horizon Return Prediction v11 Kaggle')
print('='*70)
print(f'  Features: {n_features} selected from {n_raw_features} by LGB gain')
print(f'  Train: 661,574 | Test: 410,139')
print(f'  CV: {"GroupKFold("+str(CFG.N_FOLDS)+")" if USE_GROUP_CV else "KFold("+str(CFG.N_FOLDS)+")"}')
print(f'  Runtime: {total_elapsed:.0f}s = {total_elapsed/60:.1f} min')
print('-'*70)
print('  MODEL         OOF R2')
for nm, oof in zip(model_names, oofs_list):
    r2  = r2_score(y, oof)
    bar = '#' * min(50, max(0, int(r2 * 1000)))
    print(f'  {nm:<14} {r2:+.6f}  {bar}')
print('-'*70)
print('  ENSEMBLE')
print(f'  Simple avg:    {r2_sa:+.6f}')
print(f'  Optimized:     {r2_opt:+.6f}')
print(f'  FINAL ({method_str}): {final_r2:+.6f}')
print('-'*70)
print('  OVERFIT DIAGNOSTIC')
print(f'  LGB_Huber   avg best_iter: {np.mean(iters_huber):.0f}')
print(f'  LGB_Shallow avg best_iter: {np.mean(iters_shal):.0f}')
if CATBOOST_DONE: print(f'  CAT         avg best_iter: {np.mean(cat_iters):.0f}')
print(f'  NOTE: Train R2 >> Val R2 is expected (weak signal, high noise).')
print('-'*70)
print('  v11 vs v10 CHANGES')
print(f'  + Feat sel: Pearson corr (~13s) -> LGB gain 80rnd (~60s)')
print(f'  + N features: 150 -> 200 (more signal captured)')
print(f'  + LGB_Huber lr: 0.05 -> 0.02 (v8 value; ~1048 iter vs ~420)')
print(f'  + CatBoost restored (depth=5, od_wait=50, v8 params)')
print(f'  + Time guards: Huber={CFG.HUBER_TIME_LIMIT}s, Shal={CFG.SHAL_TIME_LIMIT}s, CAT={CFG.CAT_HARD_STOP}s')
print(f'  + NOTEBOOK_HARD_STOP={CFG.HARD_STOP_S}s (28 min)')
if CATBOOST_DONE:
    print(f'  CAT status: {len(cat_scores)}/5 folds completed')
else:
    print(f'  CAT status: excluded (< 3 folds or unavailable)')
print('-'*70)
print('  COMPLIANCE')
print('  OK Data: /kaggle/input/competitions/short-horizon-return-prediction-challenge-by-i-rage/')
print('  OK No internet, no pip install, no external data')
print('  OK CPU only (task_type not needed — LGB uses CPU by default)')
print('  OK GroupKFold on CV_GROUP')
print('  OK All FE: row-wise only, zero cross-sample operations')
print('  OK Standardization: train statistics only')
print('  OK Feature selection: train data only')
print('  OK No cross-sample test operations')
runtime_ok = total_elapsed < 1800
status = 'OK' if runtime_ok else 'FAIL'
print(f'  {status} Runtime {total_elapsed/60:.1f} min ({"within" if runtime_ok else "EXCEEDS"} 30-min limit)')
print('='*70)

if   final_r2 > 0.005: print('EXCEPTIONAL — top leaderboard')
elif final_r2 > 0.003: print('STRONG — prize range')
elif final_r2 > 0.002: print('GOOD — above majority')
elif final_r2 > 0.001: print('MODERATE — above baseline')
elif final_r2 > 0:     print('WEAK — positive signal')
else:                   print('NO SIGNAL')

print()
print('NOTEBOOK COMPLETE v11 Kaggle')
if np.mean(iters_huber) < 100:
    print('WARNING: LGB_Huber avg iter < 100. lr=0.02 should give ~1000 iter.')
if CATBOOST_DONE:
    print(f'CAT: {len(cat_scores)}/5 folds, avg_iter={np.mean(cat_iters):.0f}')
else:
    print('CAT: not included in ensemble (time budget or unavailable)')
print(f'Elapsed: {time.time()-NOTEBOOK_START:.0f}s = {(time.time()-NOTEBOOK_START)/60:.1f} min')

## Submission Checklist (v10)

| Rule | Status | Detail |
|---|---|---|
| Data path | PASS | `/kaggle/input/competitions/short-horizon-return-prediction-challenge-by-i-rage/` |
| No internet | PASS | No requests, urllib, pip install |
| CPU only | PASS | LGB default is CPU; no task_type override needed |
| No cross-sample test ops | PASS | All FE row-wise; no KNN/ranking/cluster on test |
| GroupKFold on CV_GROUP | PASS | 70 groups → 5 folds |
| Normalization: train stats | PASS | `tr_mean`, `tr_std` from train only |
| Feature selection: train only | PASS | `|corr(X_train, y_train)|` via numpy |
| Runtime guard | PASS | Per-model limits + hard stop at 23 min |
| submission.csv validation | PASS | 6-guard assert before save |
| Row count | PASS | 410,139 rows enforced by assert |